# ИДЗ. Проверка гипотезы о независимости двух случайных величин, измеренных в порядковой шкале

Сформулировать реальную задачу, решение которой сводится к проверке статистической гипотезы о независимости двух случайных величин, измеренных в порядковой шкале.


In [3]:
import pandas as pd 
import numpy as np 

import seaborn as sns 
import matplotlib.pyplot as plt 

import math
from math import sqrt

import kagglehub
from kagglehub import KaggleDatasetAdapter

import warnings
warnings.filterwarnings('ignore')

## Описание данных

Для анализа используется датасет Student Performance Dataset.

Источник данных (Kaggle): https://www.kaggle.com/datasets/larsen0966/student-performance-data-set

Датасет содержит информацию об успеваемости школьников и их дополнительных характеристиках, в том числе итоговых результатах обучения.

В рамках исследования рассматриваются два порядковых признака:

* G1 — оценка за первый учебный период (модуль);
* G3 — итоговая оценка за курс.

In [4]:
df = pd.read_csv('student-por.csv.xls')
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [15]:
df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
count,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000,649.000000
mean,16.744222,2.514638,2.306626,1.568567,1.930663,0.221880,3.930663,3.180277,3.184900,1.502311,2.280431,3.536210,3.659476,11.399076,11.570108,11.906009
std,1.218138,1.134552,1.099931,0.748660,0.829510,0.593235,0.955717,1.051093,1.175766,0.924834,1.284380,1.446259,4.640759,2.745265,2.913639,3.230656
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,16.000000,2.000000,1.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,2.000000,0.000000,10.000000,10.000000,10.000000
50%,17.000000,2.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,4.000000,2.000000,11.000000,11.000000,12.000000
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,6.000000,13.000000,13.000000,14.000000
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,32.000000,19.000000,19.000000,19.000000


## Постановка статистической задачи

Цель исследования — определить, существует ли статистическая зависимость между текущей и итоговой успеваемостью студентов.

Пусть:

* X — ранг студента по показателю G1;
* Y — ранг студента по показателю G3.

Требуется проверить гипотезу о независимости случайных величин X и Y.

### Основная гипотеза

H₀: случайные величины X и Y независимы

Иными словами можно сформулировать, что положение студента в ранговом ряду по текущей успеваемости не связано с его положением в ранговом ряду по итоговой оценке.

### Альтернативная гипотеза

H₁: между случайными величинами X и Y существует статистическая зависимость.

Уровень значимости: α = 0.05.

In [9]:
# Отбор признаков
X = list(df["G1"])
Y = list(df["G3"])

# Ранги
rank_x = [0] * len(X)
sorted_x = sorted(X)

for i in range(len(X)):
    for j in range(len(sorted_x)):
        if X[i] == sorted_x[j]:
            rank_x[i] = j + 1
            break

# Ранги для Y
rank_y = [0] * len(Y)
sorted_y = sorted(Y)

for i in range(len(Y)):
    for j in range(len(sorted_y)):
        if Y[i] == sorted_y[j]:
            rank_y[i] = j + 1
            break

n = len(X)

## Теоретические сведения

### Критерий Спирмена

Коэффициент ранговой корреляции Спирмена является непараметрическим аналогом коэффициента корреляции Пирсона и используется для анализа монотонной зависимости между порядковыми признаками.

Каждому наблюдению присваиваются ранги Ri и Si. Затем вычисляются разности рангов:

di = Ri − Si.

Коэффициент Спирмена определяется формулой:

rs = 1 − 6Σdi² / (n(n²−1)).

Если rs близок к 1, наблюдается сильная положительная зависимость.
Если rs близок к −1, наблюдается сильная отрицательная зависимость.
Если rs близок к 0, зависимость отсутствует.

Для проверки гипотезы используется статистика Стьюдента:

t = rs·√((n−2)/(1−rs²)).

In [12]:
sum_d2 = 0

for i in range(n):
    d = rank_x[i] - rank_y[i]
    sum_d2 += d * d

rs = 1 - (6 * sum_d2) / (n * (n**2 - 1))

print("Σd² =", sum_d2)
print("rs =", rs)

t = rs * math.sqrt((n - 2) / (1 - rs**2))

print("t =", t)

alpha = 0.05

t_critical = 1.96

if abs(t) > t_critical:
    print("H0 отвергается")
else:
    print("Нет оснований отвергать H0")

Σd² = 5374788
rs = 0.8820278403329251
t = 47.613668837541326
H0 отвергается


### Критерий Кендалла

Коэффициент Кендалла основан на сравнении всех возможных пар наблюдений.

Пара называется согласованной, если порядок объектов по обоим признакам совпадает.

Пара называется несогласованной, если порядок объектов различается.

Обозначим:

* P — число согласованных пар;
* Q — число несогласованных пар.

Тогда коэффициент Кендалла вычисляется по формуле:

τ = (P−Q)/(n(n−1)/2).

Для больших выборок используется статистика:

Z = τ / √(2(2n+5)/(9n(n−1))).

При справедливости H₀ статистика имеет асимптотически стандартное нормальное распределение.

In [14]:
P = 0
Q = 0

for i in range(n):
    for j in range(i + 1, n):

        x_diff = rank_x[j] - rank_x[i]
        y_diff = rank_y[j] - rank_y[i]

        product = x_diff * y_diff

        if product > 0:
            P += 1

        elif product < 0:
            Q += 1

pairs = n * (n - 1) / 2

tau = (P - Q) / pairs

print("Согласованных пар =", P)
print("Несогласованных пар =", Q)
print("tau =", tau)

z = tau / math.sqrt(
    (2 * (2 * n + 5))
    /
    (9 * n * (n - 1))
)

print("z =", z)

if abs(z) > 1.96:
    print("H0 отвергается")
else:
    print("Нет оснований отвергать H0")

Согласованных пар = 158466
Несогласованных пар = 13791
tau = 0.6880243109056668
z = 26.2209054437284
H0 отвергается


## Вывод

В ходе исследования была проверена гипотеза о независимости двух случайных величин, измеренных в порядковой шкале: текущей успеваемости студентов (оценка G1) и итоговой успеваемости (оценка G3).

В результате применения критерия Спирмена было получено значение коэффициента ранговой корреляции

$$
r_s = 0.882.
$$

Полученное значение близко к единице, что свидетельствует о наличии сильной положительной связи между рассматриваемыми признаками. Иными словами, студенты, имеющие высокие оценки в начале обучения, как правило, демонстрируют высокие итоговые результаты.

Для проверки статистической значимости коэффициента была вычислена t-статистика

$$
t = 47.61.
$$

Полученное значение значительно превышает критическое значение для уровня значимости (\alpha = 0.05). Следовательно, нулевая гипотеза о независимости признаков отвергается.

Таким образом, результаты исследования позволяют сделать вывод о существовании статистически значимой положительной зависимости между текущей и итоговой успеваемостью студентов. Текущие результаты обучения являются хорошим предиктором итоговой оценки, а ранговое положение студента по показателю G1 тесно связано с его ранговым положением по показателю G3.

Следовательно, на уровне значимости 0.05 имеются достаточные статистические основания отвергнуть гипотезу о независимости исследуемых случайных величин и принять альтернативную гипотезу о наличии зависимости между ними.

